# 🎬 Cinematic Engine V17 — Enterprise AI Platform

### الخطوات بالترتيب — شغّل كل cell بالترتيب ✅

1. `Runtime → Change runtime type → T4 GPU → Save`
2. شغّل **Cell 1** — تحميل المشروع وتثبيت الـ packages
3. شغّل **Cell 2** — تحميل الـ Engine
4. شغّل **Cell 3** — تشغيل الـ Bridge
5. افتح الـ Dashboard في المتصفح
6. غيّر `WS_URL` في الـ Dashboard للـ ngrok URL اللي هيظهر في Cell 3


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — تحميل المشروع وتثبيت الـ packages            ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, os, sys, torch

# ── GPU Check ────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU — Runtime → Change runtime type → T4 GPU')
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ GPU: {gpu} | VRAM: {vram:.1f} GB')

# ── تحميل المشروع من GitHub ──────────────────────────────────
REPO_URL = 'https://github.com/michelluke1984-cyber/cinematic-engine'
REPO_DIR = '/content/cinematic-engine'

if os.path.exists(REPO_DIR):
    print('↷ المشروع موجود — بيتحدّث...')
    os.system(f'cd {REPO_DIR} && git pull -q')
else:
    print('⬇ تحميل المشروع...')
    ret = os.system(f'git clone -q {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('فشل تحميل المشروع — تأكد من الاتصال بالنت')

# التأكد من الملفات المهمة
for fname in ['cinematic_engine_v16_pro.py', 'cev17_backend.py']:
    path = f'{REPO_DIR}/{fname}'
    if os.path.exists(path):
        print(f'  ✓ {fname}')
    else:
        raise FileNotFoundError(f'✗ {fname} — مش موجود في GitHub!')

# نقل لـ working directory
os.chdir(REPO_DIR)
print(f'✓ Working directory: {REPO_DIR}')

# ── تثبيت الـ packages ───────────────────────────────────────
def install(cmd, label):
    print(f'  {label}…', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('✓' if r.returncode == 0 else f'⚠ {r.stderr[:60]}')

print('\n📦 تثبيت الـ packages:')
install('pip install -q diffusers transformers accelerate xformers einops', 'diffusers')
install('pip install -q gfpgan realesrgan nltk sentencepiece tqdm',         'GFPGAN + NLTK')
install('pip install -q controlnet-aux open-clip-torch',                    'ControlNet')
install('pip install -q "gradio>=4.0.0"',                                   'Gradio')
install('pip install -q bitsandbytes Pillow',                               'bitsandbytes')
install('pip install -q insightface onnxruntime-gpu',                       'InsightFace')
install('pip install -q websockets nest_asyncio',                           'WebSocket')
install('pip install -q pyngrok',                                           'ngrok')

# NLTK data
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print('  NLTK data ✓')

# تحميل model weights
print('\n⬇ تحميل الـ model weights:')
models = {
    'GFPGANv1.3.pth':           'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    'realesr-general-x4v3.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
}
for fname, url in models.items():
    if os.path.exists(fname):
        print(f'  ↷ {fname} موجود')
    else:
        print(f'  ⬇ {fname}…', end=' ', flush=True)
        subprocess.run(f'wget -q -O {fname} {url}', shell=True)
        size = os.path.getsize(fname)/1e6 if os.path.exists(fname) else 0
        print(f'✓ ({size:.0f}MB)')

print('\n' + '='*50)
print('  ✓ CELL 1 مكتمل — شغّل Cell 2')
print('='*50)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — تحميل الـ V16 Engine                          ║
# ╚══════════════════════════════════════════════════════════╝

import importlib.util, sys, os, torch

# التأكد من working directory
REPO_DIR = '/content/cinematic-engine'
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)
    print(f'✓ Changed to: {REPO_DIR}')

ENGINE_PATH = os.path.join(REPO_DIR, 'cinematic_engine_v16_pro.py')

if not os.path.exists(ENGINE_PATH):
    raise FileNotFoundError(
        f'✗ الملف مش موجود: {ENGINE_PATH}\n'
        f'  شغّل Cell 1 الأول'
    )

print(f'✓ Engine file: {ENGINE_PATH}')
print('Loading V16 engine...')

# ── الخطوة 1: تحميل الـ module وتنفيذه ──────────────────────
spec  = importlib.util.spec_from_file_location('cev16', ENGINE_PATH)
cev16 = importlib.util.module_from_spec(spec)
sys.modules['cev16'] = cev16
spec.loader.exec_module(cev16)  # ← لازم يتنفّذ الأول قبل أي export

# ── الخطوة 2: export الـ classes المهمة ─────────────────────
NEEDED = [
    'CinematicEngineV16', 'ModelMode', 'SpeedMode', 'PhysicsMode',
    'GenerationConfig', 'Config', 'VRAMManager', 'PipelineManager',
    'SmartCache', 'CinematicLogger', 'logger',
]
exported = []
for name in NEEDED:
    obj = getattr(cev16, name, None)
    if obj is not None:
        globals()[name] = obj
        sys.modules['__main__'].__dict__[name] = obj
        exported.append(name)

# export الباقي
for k, v in vars(cev16).items():
    if not k.startswith('_') and k not in exported:
        globals()[k] = v
        sys.modules['__main__'].__dict__[k] = v

# ── الخطوة 3: التأكد إن CinematicEngineV16 موجودة ───────────
if 'CinematicEngineV16' not in globals():
    raise RuntimeError(
        '✗ CinematicEngineV16 مش موجودة في الملف\n'
        '  تأكد إن cinematic_engine_v16_pro.py هو الملف الصح'
    )

print(f'✓ Exported: {", ".join(exported[:5])}...')

# ── الخطوة 4: إنشاء الـ engine ──────────────────────────────
print('[ENGINE] Initialising CinematicEngineV16…')
engine = CinematicEngineV16()

# حفظ engine في كل الـ scopes عشان Cell 3 يلاقيه
globals()['engine'] = engine
sys.modules['__main__'].__dict__['engine'] = engine

print('[ENGINE] ✓ Ready')
print(f'[ENGINE] GPU: {torch.cuda.get_device_name(0)}')
print(f'[ENGINE] VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print('\n' + '='*50)
print('  ✓ CELL 2 مكتمل — شغّل Cell 3')
print('='*50)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — تشغيل الـ WebSocket Bridge                    ║
# ║  اضغط ⬛ لو عايز توقفه                                  ║
# ╚══════════════════════════════════════════════════════════╝

import importlib.util, sys, os, nest_asyncio

# ضروري في Colab عشان asyncio يشتغل صح
nest_asyncio.apply()

# ── إعدادات ───────────────────────────────────────────────────
USE_NGROK   = True
NGROK_TOKEN = 'L3NYYCRD5VTGCOML4S2EBWAMDMN6KI2K'

# ── working directory ─────────────────────────────────────────
REPO_DIR = '/content/cinematic-engine'
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

# ── تحرير البورت لو كان مشغول ────────────────────────────────
os.system("fuser -k 7860/tcp 2>/dev/null || true")

# ── التأكد إن engine محمّل ────────────────────────────────────
_engine = globals().get('engine') or sys.modules.get('__main__', {}).__dict__.get('engine')
if _engine is None:
    raise RuntimeError('✗ Engine مش محمّل — شغّل Cell 2 الأول ثم ارجع لهنا')
print(f'✓ Engine found: {type(_engine).__name__}')

# ── تحميل الـ bridge ──────────────────────────────────────────
BRIDGE_PATH = os.path.join(REPO_DIR, 'cev17_backend.py')
if not os.path.exists(BRIDGE_PATH):
    raise FileNotFoundError(f'✗ {BRIDGE_PATH} مش موجود')

spec   = importlib.util.spec_from_file_location('bridge', BRIDGE_PATH)
bridge = importlib.util.module_from_spec(spec)
sys.modules['bridge'] = bridge
spec.loader.exec_module(bridge)
print('✓ Bridge loaded')

# ── ngrok ─────────────────────────────────────────────────────
if USE_NGROK and NGROK_TOKEN:
    from pyngrok import conf, ngrok as _ngrok
    conf.get_default().auth_token = NGROK_TOKEN
    # افتح الـ tunnel مسبقاً عشان نعرض الـ URL قبل ما يبدأ الـ bridge
    tunnel = _ngrok.connect(7860, 'tcp')
    ws_url = tunnel.public_url.replace('tcp://', 'ws://')
    print('\n' + '='*55)
    print('  🌐 NGROK PUBLIC URL:')
    print(f'  {ws_url}')
    print('='*55)
    print('  ⚠ في الـ Dashboard افتح الـ HTML وغيّر السطر ده:')
    print(f'  const WS_URL = \'ws://localhost:7860/ws\';')
    print('  لـ:')
    print(f'  const WS_URL = \'{ws_url}\';')
    print('='*55)
    print()

# ── تشغيل الـ bridge ─────────────────────────────────────────
print('🚀 Bridge شغّال — اضغط ⬛ عشان توقف')
bridge.run_bridge(engine=_engine, use_ngrok=False)  # ngrok شغّال بالفعل فوق

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — توليد صورة مباشرة بدون Dashboard             ║
# ║  شغّله بعد Cell 2 فقط                                   ║
# ╚══════════════════════════════════════════════════════════╝

# تحميل FLUX SCHNELL
engine.load_models(model_mode=ModelMode.FLUX_SCHNELL)

# توليد مشهد
cfg = GenerationConfig(
    scene_text = 'A weary detective stands in a rain-soaked alley, neon lights reflecting in puddles',
    camera     = 'Low Angle',
    lighting   = 'Neon Cyberpunk',
    style      = 'Cinematic',
)
print('🎬 Generating...')
img, meta = engine.generate_scene(cfg)
print(f'✓ Done')
img